# Tutorial 1 — Images and the gold-standard annotation

This notebook opens one image set from the ecdna-bench resource (BioImage Archive
**S-BIAD4097**) and shows how the gold-standard annotation becomes the objects and
counts used everywhere in the benchmark.

You will:

1. load the probe-channel RGB image, the DAPI image, the region-of-interest (ROI)
   mask and the gold-standard annotation (points and rendered mask);
2. count gold-standard objects exactly as the benchmark does
   (8-connected components of at least 3 px);
3. see why the object count can be lower than the number of annotated points;
4. turn your own point annotations into a gold-standard mask.

**Data.** Set `ECDNA_DATA_ROOT` to the folder that contains `images/` (see
`docs/TUTORIAL_EXTERNAL.md`, section 3). Without data the notebook runs on a
synthetic stand-in so you can check your installation.

**Naming.** In file and folder names, `gt` stands for the gold-standard annotation
(for example `images/gt_image/`); those names match the deposited archive record.

In [ ]:
import os, sys, time
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "configs" / "default.yaml").exists():
            return p
    raise RuntimeError("Run this notebook from inside the ecdna-bench repository.")

REPO = find_repo_root()
# Where you downloaded the BioImage Archive files (the folder that contains images/).
DATA_ROOT = Path(os.environ.get("ECDNA_DATA_ROOT", Path.home() / "ecdna_data")).expanduser()
if (DATA_ROOT / "Files" / "images").is_dir():
    DATA_ROOT = DATA_ROOT / "Files"
HAVE_DATA = (DATA_ROOT / "images" / "gt_image").is_dir()
print("repository :", REPO)
print("data folder:", DATA_ROOT, "(found)" if HAVE_DATA else "(not found: the notebook runs on a small synthetic example)")

try:
    import ecdna_bench
    print("ecdna_bench:", Path(ecdna_bench.__file__).parent)
except ImportError:
    sys.path.insert(0, str(REPO / "src"))
    import ecdna_bench
    print("ecdna_bench imported from", REPO / "src")

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

NATIVE_SHAPE = (2048, 2448)
ANCHOR_UID = "ncih2170_facs_fish_0723_low_her2_52"   # the example image used in the paper

SUFFIXES = ("", "_pred_roi", "_predicted_roi", "_pred", "_roi", "_mask")

def find_file(folder, uid):
    """The file in `folder` named after the unique identifier (any image extension)."""
    folder = Path(folder) if folder else None
    if folder is None or not folder.is_dir():
        return None
    for suffix in SUFFIXES:
        hits = sorted(p for p in folder.rglob(uid + suffix + ".*")
                      if p.stem == uid + suffix
                      and p.suffix.lower() in {".tif", ".tiff", ".png", ".npy", ".npz"})
        if hits:
            return hits[0]
    return None

def read_image(path, color=False):
    flag = cv2.IMREAD_COLOR if color else cv2.IMREAD_UNCHANGED
    img = cv2.imread(str(path), flag)
    if img is None:
        import tifffile
        img = tifffile.imread(str(path))
    if color:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else np.dstack([img] * 3)
    elif img.ndim == 3:
        img = img.max(axis=2)
    return img

def render_diamonds(points_rc, shape, radius=2):
    """Render (row, col) points as diamonds (|dy| + |dx| <= radius; 13 px for radius 2)."""
    mask = np.zeros(shape, np.uint8)
    offsets = [(dy, dx) for dy in range(-radius, radius + 1)
               for dx in range(-radius, radius + 1) if abs(dy) + abs(dx) <= radius]
    for r, c in np.asarray(points_rc, dtype=int):
        for dy, dx in offsets:
            y, x = r + dy, c + dx
            if 0 <= y < shape[0] and 0 <= x < shape[1]:
                mask[y, x] = 255
    return mask

def synthetic_example(seed=0, n=60):
    """A small stand-in image set, used only when the data folder is missing."""
    rng = np.random.default_rng(seed)
    pts = np.stack([rng.integers(800, 1250, n), rng.integers(1000, 1450, n)], 1)
    gs = render_diamonds(pts, NATIVE_SHAPE)
    rgb = np.full(NATIVE_SHAPE + (3,), 8, np.uint8)
    for r, c in pts:
        cv2.circle(rgb, (int(c), int(r)), 2, (40, 220, 60), -1)
    roi = np.zeros(NATIVE_SHAPE, np.uint8); roi[700:1350, 900:1550] = 255
    return {"uid": "synthetic_example", "rgb": rgb, "dapi": rgb[:, :, 2].copy(),
            "gs": gs, "roi": roi, "points": pts}

def load_image_set(uid=None):
    """Load RGB, DAPI, gold-standard mask, points and ROI for one image set."""
    if not HAVE_DATA:
        return synthetic_example()
    img_dir = DATA_ROOT / "images"
    if uid is None:
        uid = ANCHOR_UID if find_file(img_dir / "gt_image", ANCHOR_UID) else \
            sorted(p.stem for p in (img_dir / "gt_image").iterdir())[0]
    out = {"uid": uid}
    for key, sub, color in [("rgb", "rgb", True), ("dapi", "dapi", False),
                            ("gs", "gt_image", False), ("roi", "roi_mask", False)]:
        p = find_file(img_dir / sub, uid)
        out[key] = read_image(p, color=color) if p else None
    p = find_file(img_dir / "gt_coords", uid)
    out["points"] = np.load(p, allow_pickle=True) if p else None
    return out

## 1. Load one image set

In [ ]:
sample = load_image_set()          # or load_image_set("snu16_...") for a specific image set
uid = sample["uid"]
print("image set:", uid)
for key in ("rgb", "dapi", "gs", "roi"):
    arr = sample[key]
    print(f"  {key:<5}", None if arr is None else f"{arr.shape} {arr.dtype} values {arr.min()}..{arr.max()}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(sample["rgb"]); axes[0].set_title("probe channel (RGB)")
if sample["dapi"] is not None:
    axes[1].imshow(sample["dapi"], cmap="gray"); axes[1].set_title("DAPI")
overlay = sample["rgb"].copy()
if sample["roi"] is not None:
    edge = cv2.morphologyEx((sample["roi"] > 0).astype(np.uint8), cv2.MORPH_GRADIENT, np.ones((9, 9), np.uint8))
    overlay[edge > 0] = (255, 255, 0)
overlay[sample["gs"] > 0] = (255, 0, 255)
axes[2].imshow(overlay); axes[2].set_title("gold standard (magenta) and ROI (yellow)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

## 2. Count gold-standard objects the way the benchmark does

The benchmark never counts annotation points directly. It renders every point as a
small diamond, then counts **8-connected components with an area of at least 3 px**.
The same function is applied to every prediction mask, so both sides are counted
identically.

In [ ]:
from ecdna_bench.evaluation.objects import objects_from_mask

gs_objects = objects_from_mask(sample["gs"], min_area=3, connectivity=8, attach_mask=False)
n_points = None if sample["points"] is None else int(np.asarray(sample["points"]).shape[0])
print("gold-standard objects (components >= 3 px):", len(gs_objects))
print("annotated points in the coordinate file    :", n_points)
areas = np.array([o["area"] for o in gs_objects])
print("object areas (px): median", np.median(areas), "| max", areas.max())

## 3. Why the two counts can differ

Two annotation points that lie within a few pixels of each other produce touching
diamonds, and touching diamonds form one connected component. The object count is
therefore at most the point count. The per-image difference is released for every
image (`coord_count_npy` versus `ecDNA_gt` in the benchmark table).

The cell below renders the points again and compares the result with the deposited
mask. Point files store two coordinates per point; the order is checked rather than
assumed.

In [ ]:
try:
    pts = None if sample["points"] is None else np.asarray(sample["points"], dtype=float)[:, :2]
except (TypeError, ValueError, IndexError) as exc:
    pts = None
    print("coordinate file has an unexpected layout:", exc)
if pts is not None:
    best = None
    for name, rc in [("(row, col)", pts), ("(x, y)", pts[:, ::-1])]:
        rendered = render_diamonds(np.rint(rc), sample["gs"].shape)
        inter = np.logical_and(rendered > 0, sample["gs"] > 0).sum()
        union = np.logical_or(rendered > 0, sample["gs"] > 0).sum()
        iou = inter / union if union else 1.0
        print(f"points read as {name:<11}: pixel IoU with the deposited mask = {iou:.3f}")
        if best is None or iou > best[1]:
            best = (name, iou, rendered)
    merged = len(pts) - len(objects_from_mask(best[2], attach_mask=False))
    print(f"points that merged into a neighbor's component: {merged}")
else:
    print("no usable coordinate file for this image set")

## 4. Your own annotations

Annotate points in Fiji (Point tool, then *Analyze > Measure* and save the table as
CSV with X and Y columns), then render them exactly like the resource. Replace the
example below with your file.

In [ ]:
import pandas as pd

# Example: a CSV with X and Y columns in pixels (Fiji's default export).
my_points = pd.DataFrame({"X": [1200, 1210, 1300], "Y": [1000, 1004, 1100]})
# my_points = pd.read_csv("my_annotations.csv")

my_mask = render_diamonds(my_points[["Y", "X"]].to_numpy(), NATIVE_SHAPE)
print("objects in my annotation:", len(objects_from_mask(my_mask, attach_mask=False)), "from", len(my_points), "points")
# cv2.imwrite("my_image_gold_standard.png", my_mask)   # values 0 and 255, like the resource